# Clustering Listwise DPO — Smoke Test

Runs the full pipeline on **10 questions**: prepare → generate traces → process → DPO train → listwise train → evaluate.

> **Runtime:** Set to **T4 GPU** (Runtime → Change runtime type → GPU → T4).  
> Uses 4-bit QLoRA so the full pipeline fits within T4's 16 GB VRAM.
> You will need a [Hugging Face token](https://huggingface.co/settings/tokens) to download Mistral-7B.

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!nvidia-smi
!pip install -q \
    "transformers==4.45.1" \
    "trl==0.9.6" \
    "peft==0.13.2" \
    "accelerate==1.13.0" \
    "datasets==4.8.4" \
    bitsandbytes \
    pyyaml \
    tqdm
print('Done.')

In [ ]:
# ── Hugging Face login (needed for Mistral-7B) ────────────────────────────────
from huggingface_hub import login
login()  # paste your HF token when prompted

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
import os, gc, json, re, random
import torch
from tqdm import tqdm

MODEL        = "mistralai/Mistral-7B-v0.1"
N_QUESTIONS  = 10    # smoke-test slice
N_SAMPLES    = 10    # traces per question
MAX_NEW_TOKENS = 256
SEED         = 42
random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}  |  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if DEVICE == "cuda" else "No GPU — expect slow generation")

# paths (all under /content so nothing needs a Drive mount)
QUESTIONS_PATH      = "/content/questions_small.jsonl"
RAW_TRACES_PATH     = "/content/raw_traces_small.jsonl"
PROCESSED_PATH      = "/content/processed_small.jsonl"
DPO_PAIRS_PATH      = "/content/dpo_pairs_small.jsonl"
LISTWISE_PATH       = "/content/listwise_pairs_small.jsonl"
DPO_OUT             = "/content/outputs/dpo_small"
LISTWISE_OUT        = "/content/outputs/listwise_small"

for d in [DPO_OUT, LISTWISE_OUT]:
    os.makedirs(d, exist_ok=True)

print("Config ready.")

In [ ]:
# ── Utility functions (mirrors generation/code/utils.py) ─────────────────────

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

def is_correct(trace, ground_truth):
    """Ground truth appears in the last 300 chars of the trace."""
    return ground_truth.strip() in trace[-300:]

def no_similar(candidate, existing, length_window=500):
    """True when candidate differs by >=length_window chars from every existing trace."""
    for t in existing:
        if abs(len(candidate) - len(t)) < length_window:
            return False
    return True

_ERROR_PHRASES = ["error", "apolog", "i cannot", "i'm unable", "i am unable"]
def exist_error(trace):
    lowered = trace.lower()
    return any(p in lowered for p in _ERROR_PHRASES)

print("Utilities defined.")

## Step 1: Prepare questions

In [ ]:
from datasets import load_dataset

def extract_gsm8k_answer(raw_answer):
    m = re.search(r"####\s*([\d,\.]+)", raw_answer)
    if m:
        return m.group(1).replace(",", "").strip()
    return raw_answer.strip()

print("Loading GSM8K train split...")
dataset = load_dataset("openai/gsm8k", "main", split="train")

questions_small = [
    {"idx": i, "question": item["question"], "answer": extract_gsm8k_answer(item["answer"])}
    for i, item in enumerate(dataset)
][:N_QUESTIONS]

save_jsonl(questions_small, QUESTIONS_PATH)
print(f"Sliced {len(questions_small)} questions → {QUESTIONS_PATH}")

## Step 2: Generate traces (10 questions × 10 samples)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 4-bit quantization so the 7B model fits comfortably on T4 (16 GB)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL} (4-bit) for generation...")
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model_gen = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb_config, device_map="auto"
)
model_gen.eval()

def build_prompt(question, tok):
    if tok.chat_template is not None:
        messages = [{"role": "user", "content": question}]
        return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return f"Question: {question}\nAnswer:"

questions_data = load_jsonl(QUESTIONS_PATH)
raw_traces = []

for item in tqdm(questions_data, desc="generating"):
    prompt = build_prompt(item["question"], tokenizer)
    inputs = tokenizer(prompt, return_tensors="pt").to(model_gen.device)
    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model_gen.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=1.0,
            num_return_sequences=N_SAMPLES,
            pad_token_id=tokenizer.eos_token_id,
        )

    for seq in output_ids:
        trace = tokenizer.decode(seq[prompt_len:], skip_special_tokens=True)
        raw_traces.append({
            "idx": item["idx"],
            "question": item["question"],
            "answer": item["answer"],
            "trace": trace,
        })

save_jsonl(raw_traces, RAW_TRACES_PATH)
print(f"Saved {len(raw_traces)} raw traces → {RAW_TRACES_PATH}")

# free VRAM before training
del model_gen
gc.collect()
torch.cuda.empty_cache()

## Step 3: Process traces (sort correct / wrong, dedup disabled for smoke test)

In [ ]:
from collections import defaultdict

raw = load_jsonl(RAW_TRACES_PATH)
pools = defaultdict(lambda: {"question": "", "answer": "", "correct_solutions": [], "wrong_solutions": []})

for item in tqdm(raw, desc="processing"):
    idx, trace, gt = item["idx"], item["trace"], item["answer"]
    pool = pools[idx]
    pool["question"] = item["question"]
    pool["answer"]   = gt

    if is_correct(trace, gt):
        if not exist_error(trace) and no_similar(trace, pool["correct_solutions"], length_window=0):
            pool["correct_solutions"].append(trace)
    else:
        if no_similar(trace, pool["wrong_solutions"], length_window=0):
            pool["wrong_solutions"].append(trace)

results = [{"idx": idx, **pool} for idx, pool in pools.items()]
save_jsonl(results, PROCESSED_PATH)

for r in results:
    print(f"  idx={r['idx']}  correct={len(r['correct_solutions'])}  wrong={len(r['wrong_solutions'])}")
print(f"\nSaved {len(results)} questions → {PROCESSED_PATH}")

## Step 4.1: Build DPO pairs

In [ ]:
random.seed(SEED)
processed = load_jsonl(PROCESSED_PATH)
dpo_pairs = []

for item in tqdm(processed, desc="building DPO pairs"):
    combos = [(c, w) for c in item["correct_solutions"] for w in item["wrong_solutions"]]
    random.shuffle(combos)
    for chosen, rejected in combos[:3]:
        dpo_pairs.append({
            "prompt":   item["question"],
            "chosen":   chosen,
            "rejected": rejected,
        })

save_jsonl(dpo_pairs, DPO_PAIRS_PATH)
print(f"Saved {len(dpo_pairs)} DPO pairs → {DPO_PAIRS_PATH}")

if len(dpo_pairs) == 0:
    print("\n⚠️  No pairs found — the model produced no correct traces on these 10 questions.")
    print("    Try increasing N_SAMPLES or N_QUESTIONS in the Config cell and re-running from Step 2.")

## Step 4.1: Train DPO

In [ ]:
from datasets import Dataset
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training
from trl import DPOTrainer, DPOConfig

assert len(dpo_pairs) > 1, "Need at least 2 DPO pairs to train — see warning above."

print(f"Loading {MODEL} (4-bit) for DPO training...")
tokenizer_dpo = AutoTokenizer.from_pretrained(MODEL)
if tokenizer_dpo.pad_token is None:
    tokenizer_dpo.pad_token = tokenizer_dpo.eos_token

model_dpo = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb_config, device_map="auto"
)
model_dpo = prepare_model_for_kbit_training(model_dpo, use_gradient_checkpointing=True)
model_dpo.config.use_cache = False

peft_config_dpo = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)

n_eval = max(1, int(len(dpo_pairs) * 0.05))
train_ds = Dataset.from_list(dpo_pairs[n_eval:])
eval_ds  = Dataset.from_list(dpo_pairs[:n_eval])

dpo_args = DPOConfig(
    output_dir=DPO_OUT,
    beta=0.1,
    learning_rate=1e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    num_train_epochs=2,
    max_length=512,
    max_prompt_length=256,
    logging_steps=1,
    save_strategy="epoch",
    bf16=True,
    fp16=False,
    remove_unused_columns=False,
)

trainer_dpo = DPOTrainer(
    model=model_dpo,
    ref_model=None,
    args=dpo_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer_dpo,
    peft_config=peft_config_dpo,
)

trainer_dpo.train()
trainer_dpo.save_model(DPO_OUT)
print(f"DPO model saved → {DPO_OUT}")

del trainer_dpo, model_dpo
gc.collect()
torch.cuda.empty_cache()

## Step 4.2: ListwiseTrainer class definitions

Inline of `training/listwise_trainer.py` — LIPO-λ loss (cascading softmax).

In [ ]:
import torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset
from transformers import Trainer
from dataclasses import dataclass


def get_per_sample_logps(model, input_ids, attention_mask, labels):
    logits      = model(input_ids=input_ids, attention_mask=attention_mask).logits
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    log_probs    = F.log_softmax(shift_logits, dim=-1)
    token_logps  = log_probs.gather(2, shift_labels.clamp(min=0).unsqueeze(-1)).squeeze(-1)
    resp_mask    = (shift_labels != -100).float()
    return (token_logps * resp_mask).sum(-1) / resp_mask.sum(-1).clamp(min=1)


def lipo_lambda_loss(chosen, r1, r2, r3, r4, lambdas=(1.0, 0.75, 0.5, 0.25)):
    def lse(*s):
        return torch.logsumexp(torch.stack(list(s), dim=-1), dim=-1)
    l1 = -(chosen - lse(chosen, r1, r2, r3, r4))
    l2 = -(r1     - lse(r1, r2, r3, r4))
    l3 = -(r2     - lse(r2, r3, r4))
    l4 = -(r3     - lse(r3, r4))
    lm = lambdas
    return (lm[0]*l1 + lm[1]*l2 + lm[2]*l3 + lm[3]*l4).mean()


class ListwiseDataset(TorchDataset):
    KEYS = ["chosen", "rejected1", "rejected2", "rejected3", "rejected4"]

    def __init__(self, data, tokenizer, max_length=1024):
        self.data      = data
        self.tokenizer = tokenizer
        self.max_len   = max_length

    def _encode(self, prompt, response):
        p_ids = self.tokenizer.encode(prompt,   add_special_tokens=True)
        r_ids = self.tokenizer.encode(response, add_special_tokens=False) + [self.tokenizer.eos_token_id]
        ids    = (p_ids + r_ids)[:self.max_len]
        labels = ([-100] * len(p_ids) + r_ids)[:self.max_len]
        return {
            "input_ids":      torch.tensor(ids,              dtype=torch.long),
            "attention_mask": torch.tensor([1]*len(ids),     dtype=torch.long),
            "labels":         torch.tensor(labels,           dtype=torch.long),
        }

    def __len__(self):  return len(self.data)

    def __getitem__(self, idx):
        item, out = self.data[idx], {}
        for key in self.KEYS:
            for k, v in self._encode(item["prompt"], item[key]).items():
                out[f"{key}_{k}"] = v
        return out


@dataclass
class ListwiseDataCollator:
    pad_token_id: int

    def __call__(self, features):
        batch = {}
        for key in ["chosen", "rejected1", "rejected2", "rejected3", "rejected4"]:
            ids   = [f[f"{key}_input_ids"]      for f in features]
            masks = [f[f"{key}_attention_mask"]  for f in features]
            lbls  = [f[f"{key}_labels"]          for f in features]
            ml = max(x.size(0) for x in ids)
            def pad(ts, v):
                return torch.stack([F.pad(t, (0, ml - t.size(0)), value=v) for t in ts])
            batch[f"{key}_input_ids"]      = pad(ids,   self.pad_token_id)
            batch[f"{key}_attention_mask"] = pad(masks, 0)
            batch[f"{key}_labels"]         = pad(lbls,  -100)
        return batch


class ListwiseTrainer(Trainer):
    def __init__(self, ref_model, beta, lambdas, **kwargs):
        super().__init__(**kwargs)
        self.ref_model = ref_model
        self.beta      = beta
        self.lambdas   = lambdas
        if self.ref_model is not None:
            self.ref_model.eval()
            for p in self.ref_model.parameters():
                p.requires_grad_(False)

    def _logps(self, model, key, inputs):
        dev = next(model.parameters()).device
        return get_per_sample_logps(
            model,
            inputs[f"{key}_input_ids"].to(dev),
            inputs[f"{key}_attention_mask"].to(dev),
            inputs[f"{key}_labels"].to(dev),
        )

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        keys = ["chosen", "rejected1", "rejected2", "rejected3", "rejected4"]
        policy = {k: self._logps(model, k, inputs) for k in keys}
        with torch.no_grad():
            if self.ref_model is None:
                with model.disable_adapter():
                    ref = {k: self._logps(model, k, inputs) for k in keys}
            else:
                ref = {k: self._logps(self.ref_model, k, inputs) for k in keys}
        scores = {k: self.beta * (policy[k] - ref[k]) for k in keys}
        loss = lipo_lambda_loss(
            scores["chosen"],
            scores["rejected1"], scores["rejected2"],
            scores["rejected3"], scores["rejected4"],
            lambdas=self.lambdas,
        )
        return (loss, None) if return_outputs else loss


print("ListwiseTrainer defined.")

## Step 4.2: Build listwise pairs

In [ ]:
def rank_by_length(traces):
    return sorted(traces, key=len)

processed = load_jsonl(PROCESSED_PATH)
listwise_pairs, skipped = [], 0

for item in tqdm(processed, desc="building listwise pairs"):
    corrects = item["correct_solutions"][:5]
    wrongs   = item["wrong_solutions"][:5]
    if len(corrects) < 1 or len(wrongs) < 4:
        skipped += 1
        continue
    chosen        = rank_by_length(corrects)[0]
    ranked_wrongs = rank_by_length(wrongs)[:4]
    listwise_pairs.append({
        "prompt":    item["question"],
        "chosen":    chosen,
        "rejected1": ranked_wrongs[0],
        "rejected2": ranked_wrongs[1],
        "rejected3": ranked_wrongs[2],
        "rejected4": ranked_wrongs[3],
    })

if skipped:
    print(f"Skipped {skipped} questions (need >=1 correct and >=4 wrong traces)")

save_jsonl(listwise_pairs, LISTWISE_PATH)
print(f"Saved {len(listwise_pairs)} listwise pairs → {LISTWISE_PATH}")

if len(listwise_pairs) == 0:
    print("\n⚠️  No listwise pairs — not enough diverse traces. Try increasing N_SAMPLES and re-running from Step 2.")

## Step 4.2: Train listwise (LIPO-λ)

In [ ]:
from transformers import TrainingArguments
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

assert len(listwise_pairs) > 1, "Need at least 2 listwise pairs — see warning above."

print(f"Loading {MODEL} (4-bit) for listwise training...")
tokenizer_lw = AutoTokenizer.from_pretrained(MODEL)
if tokenizer_lw.pad_token is None:
    tokenizer_lw.pad_token = tokenizer_lw.eos_token

base_model_lw = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb_config, device_map="auto"
)
base_model_lw = prepare_model_for_kbit_training(base_model_lw, use_gradient_checkpointing=True)
base_model_lw.config.use_cache = False

peft_config_lw = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)
model_lw = get_peft_model(base_model_lw, peft_config_lw)
model_lw.print_trainable_parameters()

n_eval      = max(1, int(len(listwise_pairs) * 0.05))
train_data  = listwise_pairs[n_eval:]
eval_data   = listwise_pairs[:n_eval]
train_ds_lw = ListwiseDataset(train_data, tokenizer_lw, max_length=512)
eval_ds_lw  = ListwiseDataset(eval_data,  tokenizer_lw, max_length=512)
collator_lw = ListwiseDataCollator(pad_token_id=tokenizer_lw.pad_token_id)

lw_args = TrainingArguments(
    output_dir=LISTWISE_OUT,
    learning_rate=1e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    num_train_epochs=2,
    logging_steps=1,
    save_strategy="epoch",
    bf16=True,
    fp16=False,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

trainer_lw = ListwiseTrainer(
    ref_model=None,
    beta=0.1,
    lambdas=(1.0, 0.75, 0.5, 0.25),
    model=model_lw,
    args=lw_args,
    train_dataset=train_ds_lw,
    eval_dataset=eval_ds_lw,
    data_collator=collator_lw,
    tokenizer=tokenizer_lw,
)

trainer_lw.train()
trainer_lw.save_model(LISTWISE_OUT)
print(f"Listwise model saved → {LISTWISE_OUT}")

del trainer_lw, model_lw, base_model_lw
gc.collect()
torch.cuda.empty_cache()

## Step 5: Evaluate both models on GSM8K test

In [ ]:
from peft import PeftModel

def extract_answer(text):
    m = re.search(r"####\s*([\d,\.]+)", text)
    if m:
        return m.group(1).replace(",", "").strip()
    nums = re.findall(r"\b\d[\d,\.]*\b", text)
    return nums[-1].replace(",", "") if nums else None

def answer_correct(pred_text, gt):
    pred = extract_answer(pred_text)
    gt_  = extract_answer(gt) or gt.strip()
    if pred is None:
        return False
    try:
        return abs(float(pred) - float(gt_)) < 1e-6
    except ValueError:
        return pred == gt_

def evaluate_model(model_path, label, test_dataset, n_examples=100):
    """Load a LoRA-trained model and evaluate on the first n_examples of the test set."""
    print(f"\n=== Evaluating {label} ===")
    tok = AutoTokenizer.from_pretrained(MODEL)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    base = AutoModelForCausalLM.from_pretrained(
        MODEL, quantization_config=bnb_config, device_map="auto"
    )
    model = PeftModel.from_pretrained(base, model_path)
    model.eval()

    correct = 0
    subset  = list(test_dataset)[:n_examples]

    for item in tqdm(subset, desc=label):
        if tok.chat_template is not None:
            prompt = tok.apply_chat_template(
                [{"role": "user", "content": item["question"]}],
                tokenize=False, add_generation_prompt=True
            )
        else:
            prompt = f"Question: {item['question']}\nAnswer:"

        inputs = tok(prompt, return_tensors="pt").to(next(model.parameters()).device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tok.eos_token_id,
            )
        response = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

        gt = item["answer"]
        # GSM8K test stores full answer — extract the number after ####
        correct += int(answer_correct(response, gt))

    accuracy = correct / len(subset)
    print(f"{label} accuracy: {accuracy:.4f}  ({correct}/{len(subset)})")

    del model, base
    gc.collect()
    torch.cuda.empty_cache()
    return accuracy


print("Loading GSM8K test split...")
gsm_test = load_dataset("openai/gsm8k", "main", split="test")

# Evaluate on first 100 test examples (full test = 1319, slow on Colab)
N_EVAL_EXAMPLES = 100

acc_dpo      = evaluate_model(DPO_OUT,      "DPO",      gsm_test, N_EVAL_EXAMPLES)
acc_listwise = evaluate_model(LISTWISE_OUT, "Listwise", gsm_test, N_EVAL_EXAMPLES)

print("\n" + "="*40)
print(f"DPO      accuracy: {acc_dpo:.4f}")
print(f"Listwise accuracy: {acc_listwise:.4f}")
print("="*40)